In [ ]:
# ============================================================
# STAGE 4 — POST-PROCESSING & VALIDATION
# D7 — Branch B — Structural Conversion
# ============================================================
#
# Validation basis:
# - Fixed Stage 1 document-grounded reference dataset
# - Branch B parsed extraction
# - Branch B technical diagnostics
# - D7 comparison rules frozen from Validation A
# ============================================================

from google.colab import files
from pathlib import Path
from difflib import SequenceMatcher

import hashlib
import json
import math
import re
import unicodedata

import pandas as pd
from scipy.optimize import linear_sum_assignment


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D7"

DOCUMENT_NAME = (
    "UK National Audit Office — Delivering STEM "
    "(science, technology, engineering and mathematics) "
    "skills for the economy"
)

BRANCH = "B"
BRANCH_NAME = "Structural Conversion"

EXPECTED_REFERENCE_RECORD_COUNT = 59

EXPECTED_CATEGORY_COUNTS = {
    "Key fact": 10,
    "Policy context": 3,
    "Policy finding": 7,
    "Education pipeline statistic": 24,
    "Government initiative": 9,
    "Recommendation": 6
}

FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Topic",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

ALIGNMENT_BLOCK_FIELDS = [
    "Category",
    "Source Location"
]

MATCHING_IDENTITY_FIELDS = [
    "Metric",
    "Topic",
    "Statement or Section",
    "Reporting Period"
]

ALIGNMENT_IDENTITY_FIELDS = (
    ALIGNMENT_BLOCK_FIELDS
    + MATCHING_IDENTITY_FIELDS
)

PRIMARY_CORRECTNESS_FIELDS = [
    "Value",
    "Unit",
    "Qualifier"
]

BLOCK_FIELDS = ALIGNMENT_BLOCK_FIELDS

MATCH_SCORE_THRESHOLD = 0.30

MATCHING_WEIGHTS = {
    "metric": 0.50,
    "topic": 0.30,
    "statement_or_section": 0.15,
    "reporting_period": 0.05
}

OUTPUT_DIR = Path("outputs_D7_validation_branch_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH, "-", BRANCH_NAME)
print("Expected reference records:", EXPECTED_REFERENCE_RECORD_COUNT)
print("Alignment identity fields:", ALIGNMENT_IDENTITY_FIELDS)
print("Primary correctness fields:", PRIMARY_CORRECTNESS_FIELDS)


In [ ]:
# ============================================================
# 2. Upload validation inputs
# ============================================================
# Required:
#   1) D7_reference_values.csv
#   2) D7_branch_B_parsed_extraction.json
#   3) D7_branch_B_technical_diagnostics.json

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

csv_files = [
    name for name in uploaded_files
    if name.lower().endswith(".csv")
]

json_files = [
    name for name in uploaded_files
    if name.lower().endswith(".json")
]

if len(csv_files) != 1:
    raise ValueError(
        "Upload exactly one D7 Stage 1 reference CSV."
    )

if len(json_files) != 2:
    raise ValueError(
        "Upload exactly two JSON files: the Branch B parsed "
        "extraction and technical diagnostics."
    )

REFERENCE_FILE = csv_files[0]
PARSED_EXTRACTION_FILE = None
TECHNICAL_DIAGNOSTICS_FILE = None

for file_name in json_files:

    with open(file_name, "r", encoding="utf-8-sig") as f:
        obj = json.load(f)

    if not isinstance(obj, dict):
        continue

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and isinstance(obj.get("records"), list)
    ):
        PARSED_EXTRACTION_FILE = file_name

    if (
        obj.get("document_id") == DOCUMENT_ID
        and obj.get("branch") == BRANCH
        and "structurally_evaluable" in obj
        and "record_schema_valid" in obj
        and "valid_json" in obj
    ):
        TECHNICAL_DIAGNOSTICS_FILE = file_name

if PARSED_EXTRACTION_FILE is None:
    raise ValueError(
        "Could not identify the D7 Branch B parsed extraction."
    )

if TECHNICAL_DIAGNOSTICS_FILE is None:
    raise ValueError(
        "Could not identify the D7 Branch B technical diagnostics."
    )

print("Reference:", REFERENCE_FILE)
print("Parsed extraction:", PARSED_EXTRACTION_FILE)
print("Technical diagnostics:", TECHNICAL_DIAGNOSTICS_FILE)


In [ ]:
# ============================================================
# 3. Load inputs and verify identity/provenance
# ============================================================

reference_df = pd.read_csv(
    REFERENCE_FILE,
    dtype=object,
    keep_default_na=False,
    encoding="utf-8-sig"
)

def restore_csv_null(value):
    if value is None:
        return None
    if isinstance(value, str) and value == "":
        return None
    return value

for column in reference_df.columns:
    reference_df[column] = reference_df[column].map(restore_csv_null)

with open(
    PARSED_EXTRACTION_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    extraction_json = json.load(f)

with open(
    TECHNICAL_DIAGNOSTICS_FILE,
    "r",
    encoding="utf-8-sig"
) as f:
    technical_diagnostics = json.load(f)

for artefact_name, artefact in {
    "parsed extraction": extraction_json,
    "technical diagnostics": technical_diagnostics
}.items():

    if artefact.get("document_id") != DOCUMENT_ID:
        raise ValueError(
            f"Unexpected {artefact_name} document_id."
        )

    if artefact.get("branch") != BRANCH:
        raise ValueError(
            f"Unexpected {artefact_name} branch."
        )

extracted_df = pd.DataFrame(
    extraction_json["records"]
)

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

input_provenance = {
    "reference_file": REFERENCE_FILE,
    "reference_sha256": sha256_file(REFERENCE_FILE),
    "parsed_extraction_file": PARSED_EXTRACTION_FILE,
    "parsed_extraction_sha256":
        sha256_file(PARSED_EXTRACTION_FILE),
    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_FILE,
    "technical_diagnostics_sha256":
        sha256_file(TECHNICAL_DIAGNOSTICS_FILE)
}

print("Reference shape:", reference_df.shape)
print("Extraction shape:", extracted_df.shape)


In [ ]:
# ============================================================
# 4. Import Branch B technical/schema status
# ============================================================

structurally_evaluable = bool(
    technical_diagnostics.get(
        "structurally_evaluable",
        False
    )
)

schema_validity = bool(
    structurally_evaluable
)

schema_diagnostics = {
    "valid_json": bool(
        technical_diagnostics.get(
            "valid_json",
            False
        )
    ),
    "record_schema_valid": bool(
        technical_diagnostics.get(
            "record_schema_valid",
            False
        )
    ),
    "field_types_valid": bool(
        technical_diagnostics.get(
            "field_types_valid",
            False
        )
    ),
    "structurally_evaluable":
        structurally_evaluable,
    "schema_validity":
        schema_validity
}

if not structurally_evaluable:
    raise ValueError(
        "D7 Branch B output is not structurally evaluable. "
        "Content-level validation cannot proceed."
    )

print(
    json.dumps(
        schema_diagnostics,
        indent=2,
        ensure_ascii=False
    )
)


In [ ]:
# ============================================================
# 5. Verify fixed reference and prepare comparison copies
# ============================================================

if reference_df.columns.tolist() != FIELDS:
    raise ValueError(
        "D7 Stage 1 reference schema does not match "
        "the fixed field list."
    )

if len(reference_df) != EXPECTED_REFERENCE_RECORD_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_REFERENCE_RECORD_COUNT} "
        f"reference records; found {len(reference_df)}."
    )

reference_category_counts = (
    reference_df["Category"]
    .value_counts()
    .to_dict()
)

if reference_category_counts != EXPECTED_CATEGORY_COUNTS:
    raise ValueError(
        "D7 Stage 1 reference category counts "
        "do not match the fixed design."
    )

missing_extraction_fields = [
    field
    for field in FIELDS
    if field not in extracted_df.columns
]

extracted_comparison_df = extracted_df.copy(deep=True)

for field in missing_extraction_fields:
    extracted_comparison_df[field] = None

extracted_comparison_df = (
    extracted_comparison_df[FIELDS].copy()
)

reference_comparison_df = (
    reference_df[FIELDS].copy(deep=True)
)

extracted_record_count = len(extracted_comparison_df)

extraction_record_count_valid = (
    extracted_record_count
    == EXPECTED_REFERENCE_RECORD_COUNT
)

extraction_category_counts = (
    extracted_comparison_df["Category"]
    .value_counts(dropna=False)
    .to_dict()
)

extraction_category_counts_valid = (
    extraction_category_counts
    == EXPECTED_CATEGORY_COUNTS
)

content_diagnostics = {
    "reference_record_count_valid": True,
    "reference_category_counts_valid": True,
    "extraction_record_count_valid":
        bool(extraction_record_count_valid),
    "extraction_category_counts_valid":
        bool(extraction_category_counts_valid),
    "branch_B_scope_complete":
        technical_diagnostics.get("scope_complete"),
    "branch_B_content_diagnostics":
        technical_diagnostics.get("content_diagnostics")
}

print("Reference records:", len(reference_comparison_df))
print("Extracted records:", len(extracted_comparison_df))
print("Missing extraction columns:", missing_extraction_fields)
print(
    "Extraction record count matches reference:",
    extraction_record_count_valid
)


In [ ]:
# ============================================================
# 6. Comparison-only text normalisation and similarity
# ============================================================

def normalise_text(value):
    if is_null(value):
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value)
    )

    replacements = {
        "\u00a0": " ",
        "\u2007": " ",
        "\u202f": " ",
        "\u2010": "-",
        "\u2011": "-",
        "\u2012": "-",
        "\u2013": "-",
        "\u2014": "-",
        "\u2212": "-",
        "£": "gbp "
    }

    for source, target in replacements.items():
        text = text.replace(source, target)

    text = re.sub(r"\s+", " ", text)

    return text.strip().casefold()


STOPWORDS = {
    "a", "an", "and", "as", "at", "by", "for", "from",
    "in", "is", "of", "on", "or", "the", "to", "was",
    "were", "with"
}


def comparison_tokens(value):
    text = normalise_text(value)

    if text is None:
        return set()

    text = re.sub(
        r"[^a-z0-9]+",
        " ",
        text
    )

    return {
        token
        for token in text.split()
        if token and token not in STOPWORDS
    }


def sequence_similarity(first, second):
    first_text = normalise_text(first)
    second_text = normalise_text(second)

    if first_text is None and second_text is None:
        return 1.0

    if first_text is None or second_text is None:
        return 0.0

    return SequenceMatcher(
        None,
        first_text,
        second_text
    ).ratio()


def jaccard_similarity(first, second):
    first_tokens = comparison_tokens(first)
    second_tokens = comparison_tokens(second)

    if not first_tokens and not second_tokens:
        return 1.0

    if not first_tokens or not second_tokens:
        return 0.0

    return (
        len(first_tokens & second_tokens)
        / len(first_tokens | second_tokens)
    )


def text_similarity(first, second):
    return max(
        sequence_similarity(first, second),
        jaccard_similarity(first, second)
    )


In [ ]:
# ============================================================
# 7. Frozen D7 source-grounded equivalence rules
# ============================================================

# ------------------------------------------------------------
# 7.1 Helper for controlled equivalence pairs
# ------------------------------------------------------------

def make_equivalence_pair(first, second):
    return frozenset(
        {
            normalise_text(first),
            normalise_text(second)
        }
    )


def pair_is_controlled_equivalent(
    first,
    second,
    equivalence_pairs
):
    first_text = normalise_text(first)
    second_text = normalise_text(second)

    if first_text == second_text:
        return True

    pair = frozenset(
        {
            first_text,
            second_text
        }
    )

    return pair in equivalence_pairs


# ------------------------------------------------------------
# 7.2 Unit equivalence
# ------------------------------------------------------------

UNIT_EQUIVALENCE_MAP = {
    "%": "percent",
    "percent": "percent",

    "£ million": "gbp million",
    "gbp million": "gbp million"
}


def canonical_unit(value):
    text = normalise_text(value)

    if text is None:
        return None

    normalised_mapping = {
        normalise_text(key): normalise_text(target)
        for key, target in UNIT_EQUIVALENCE_MAP.items()
    }

    return normalised_mapping.get(
        text,
        text
    )


CONTROLLED_UNIT_EQUIVALENCE_PAIRS = {
    make_equivalence_pair(
        "graduates",
        "people"
    ),

    make_equivalence_pair(
        "career routes",
        "routes"
    )
}


def is_unit_equivalent(
    reference_value,
    extracted_value
):
    if (
        canonical_unit(reference_value)
        == canonical_unit(extracted_value)
    ):
        return True

    return pair_is_controlled_equivalent(
        reference_value,
        extracted_value,
        CONTROLLED_UNIT_EQUIVALENCE_PAIRS
    )


# ------------------------------------------------------------
# 7.3 Reporting-period equivalence
# ------------------------------------------------------------

PERIOD_EQUIVALENCE_MAP = {
    "between 2007 and autumn 2017":
        "2007 to autumn 2017",

    "2007 to autumn 2017":
        "2007 to autumn 2017",

    "six months later":
        "six months after graduation",

    "six months after graduation":
        "six months after graduation",

    "within six months of graduation":
        "within six months",

    "within six months":
        "within six months",

    "2016/17 compared with the previous year":
        "2016/17",

    "2016/17 versus previous year":
        "2016/17",

    "between 2011/12 and 2015/16":
        "2011/12 to 2015/16",

    "2011/12 to 2015/16":
        "2011/12 to 2015/16",

    "between 2011/12 and 2016/17":
        "2011/12 to 2016/17",

    "2011/12 to 2016/17":
        "2011/12 to 2016/17"
}


def canonical_period(value):
    text = normalise_text(value)

    if text is None:
        return None

    normalised_mapping = {
        normalise_text(key): normalise_text(target)
        for key, target in PERIOD_EQUIVALENCE_MAP.items()
    }

    return normalised_mapping.get(
        text,
        text
    )


def is_period_equivalent(
    reference_value,
    extracted_value
):
    return (
        canonical_period(reference_value)
        == canonical_period(extracted_value)
    )


# ------------------------------------------------------------
# 7.4 Qualifier equivalence
# ------------------------------------------------------------


CONTROLLED_QUALIFIER_EQUIVALENCE_PAIRS = {
    make_equivalence_pair(
        "minimum",
        "and upwards"
    )
}


def canonical_qualifier(value):
    return normalise_text(value)


def is_qualifier_equivalent(
    reference_value,
    extracted_value
):
    if (
        canonical_qualifier(reference_value)
        == canonical_qualifier(extracted_value)
    ):
        return True

    return pair_is_controlled_equivalent(
        reference_value,
        extracted_value,
        CONTROLLED_QUALIFIER_EQUIVALENCE_PAIRS
    )


# ------------------------------------------------------------
# 7.5 Statement / Section equivalence
# ------------------------------------------------------------

def canonical_statement(value):
    text = normalise_text(value)

    if text is None:
        return None

    summary_match = re.search(
        r"summary paragraph\s+(\d+)",
        text
    )

    if summary_match:
        return (
            f"summary paragraph "
            f"{summary_match.group(1)}"
        )

    recommendation_match = re.search(
        r"recommendation\s+"
        r"(\d+)\s*\(?([a-f])\)?",
        text
    )

    if recommendation_match:
        return (
            "recommendation "
            f"{recommendation_match.group(1)}"
            f"({recommendation_match.group(2)})"
        )

    return text


D7_ALLOWED_STATEMENT_BY_SOURCE = {
    normalise_text(
        "PDF page 7 — Summary paragraph 1"
    ): {
        normalise_text("Background")
    },

    normalise_text(
        "PDF page 7 — Summary paragraph 2"
    ): {
        normalise_text("Background")
    },

    normalise_text(
        "PDF page 7 — Summary paragraph 3"
    ): {
        normalise_text("Background")
    },

    normalise_text(
        "PDF page 7 — Summary paragraph 4"
    ): {
        normalise_text(
            "Government intervention"
        )
    },

    normalise_text(
        "PDF page 8 — Summary paragraph 5"
    ): {
        normalise_text(
            "Government intervention"
        )
    },

    normalise_text(
        "PDF page 8 — Summary paragraph 6"
    ): {
        normalise_text(
            "Scope and approach"
        )
    },

    normalise_text(
        "PDF page 8 — Summary paragraph 7"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 9 — Summary paragraph 8"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 9 — Summary paragraph 9"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 9 — Summary paragraph 10"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 9 — Summary paragraph 11"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 9 — Summary paragraph 12"
    ): {
        normalise_text(
            "Government's understanding of the need "
            "for enhanced STEM skills in the workforce"
        )
    },

    normalise_text(
        "PDF page 10 — Summary paragraph 13"
    ): {
        normalise_text(
            "The performance of the education pipeline "
            "in delivering STEM skills"
        )
    },

    normalise_text(
        "PDF page 10 — Summary paragraph 14"
    ): {
        normalise_text(
            "Females are underrepresented in most STEM "
            "subject areas at every stage of the STEM skills pipeline"
        )
    },

    normalise_text(
        "PDF page 10 — Summary paragraph 15"
    ): {
        normalise_text(
            "The number of people participating in STEM-related "
            "vocational courses has risen in some areas but not others"
        )
    },

    normalise_text(
        "PDF page 10 — Summary paragraph 16"
    ): {
        normalise_text(
            "Enrolments in undergraduate STEM courses "
            "have fallen slightly since 2011/12"
        )
    },

    normalise_text(
        "PDF page 11 — Summary paragraph 17"
    ): {
        normalise_text(
            "Graduate outcomes from STEM degrees"
        )
    },

    normalise_text(
        "PDF page 11 — Summary paragraph 18"
    ): {
        normalise_text(
            "The latest initiatives designed to enhance "
            "the development of STEM skills"
        )
    },

    normalise_text(
        "PDF page 11 — Summary paragraph 19"
    ): {
        normalise_text(
            "The latest initiatives designed to enhance "
            "the development of STEM skills"
        )
    },

    normalise_text(
        "PDF page 11 — Summary paragraph 20"
    ): {
        normalise_text(
            "Schools sector teacher supply initiatives"
        )
    },

    normalise_text(
        "PDF page 11 — Summary paragraph 21"
    ): {
        normalise_text(
            "Conclusion on value for money"
        )
    },

    normalise_text(
        "PDF page 12 — Recommendation 22(a)"
    ): {
        normalise_text("DfE should")
    },

    normalise_text(
        "PDF page 12 — Recommendation 22(b)"
    ): {
        normalise_text("DfE should")
    },

    normalise_text(
        "PDF page 12 — Recommendation 23(c)"
    ): {
        normalise_text("BEIS should")
    },

    normalise_text(
        "PDF page 12 — Recommendation 23(d)"
    ): {
        normalise_text("BEIS should")
    },

    normalise_text(
        "PDF page 12 — Recommendation 24(e)"
    ): {
        normalise_text(
            "DfE and other key departments should"
        )
    },

    normalise_text(
        "PDF page 12 — Recommendation 24(f)"
    ): {
        normalise_text(
            "DfE and other key departments should"
        )
    }
}


def is_statement_equivalent(
    reference_value,
    extracted_value,
    source_location
):
    if (
        canonical_statement(reference_value)
        == canonical_statement(extracted_value)
    ):
        return True

    source_key = normalise_text(
        source_location
    )

    extracted_text = normalise_text(
        extracted_value
    )

    allowed_values = (
        D7_ALLOWED_STATEMENT_BY_SOURCE.get(
            source_key,
            set()
        )
    )

    return extracted_text in allowed_values


# ------------------------------------------------------------
# 7.6 Metric equivalence
# ------------------------------------------------------------

CONTROLLED_METRIC_EQUIVALENCE_PAIRS = {
    make_equivalence_pair(
        "Female students as a share of all STEM A level examination entries",
        "Female share of all STEM A level exam entries"
    ),

    make_equivalence_pair(
        "Female students as a share of computing examination entries",
        "Female share of computing examination entries"
    ),

    make_equivalence_pair(
        "Female students as a share of physics examination entries",
        "Female share of physics examination entries"
    ),

    make_equivalence_pair(
        "Female students as a share of mathematics examination entries",
        "Female share of mathematics examination entries"
    ),

    make_equivalence_pair(
        "Female students as a share of STEM apprenticeship starts",
        "Female share of STEM apprenticeship starts"
    ),

    make_equivalence_pair(
        "Women as a share of all apprenticeship starts",
        "Female share of all apprenticeship starts"
    ),

    make_equivalence_pair(
        "Female students as a share of undergraduate STEM enrolments",
        "Female share of undergraduate STEM enrolments"
    ),

    make_equivalence_pair(
        "Women as a share of all undergraduate enrolments",
        "Female share of all undergraduate enrolments"
    ),

    make_equivalence_pair(
        "Non-apprenticeship STEM further education learning aims",
        "Non-apprenticeship STEM further education learning aims being studied"
    ),

    make_equivalence_pair(
        "Rise in enrolments across all subjects",
        "Rise in full-time degree enrolments in all subjects"
    ),

    make_equivalence_pair(
        "Overall fall in part-time degree enrolments",
        "Fall in part-time degree enrolments overall"
    ),

    make_equivalence_pair(
        "People who graduated with a STEM degree",
        "People graduating with a STEM degree"
    ),

    make_equivalence_pair(
        "STEM graduates known to be working in a STEM occupation within six months",
        "STEM graduates known to be working in a STEM occupation"
    ),

    make_equivalence_pair(
        "STEM graduates whose destinations were unknown",
        "STEM graduates whose destinations are unknown"
    ),

    make_equivalence_pair(
        "T levels are designed to improve vocational education by standardising qualifications, aligning syllabuses with employer demand and establishing clear routes into careers",
        "Clear routes into careers established by T levels"
    ),

    make_equivalence_pair(
        "Institutes of technology will target skills gaps at levels 4 and upwards, particularly in STEM areas",
        "Qualification level targeted by institutes of technology"
    ),

    make_equivalence_pair(
        "Target for recruiting additional maths and physics teachers",
        "Target for recruiting additional teachers"
    ),

    make_equivalence_pair(
        "Returning teachers recruited by the return to teaching pilot",
        "Returning teachers recruited"
    ),

    make_equivalence_pair(
        "Target for returning teachers recruited by the return to teaching pilot",
        "Recruitment target"
    ),

    make_equivalence_pair(
        "Spent on, or committed to, key STEM-specific interventions",
        "Spending on or committed to key STEM-specific interventions"
    ),

    make_equivalence_pair(
        "Graduates in STEM subjects known to be working in a STEM occupation six months later",
        "STEM graduates known to be working in a STEM occupation"
    ),

    make_equivalence_pair(
        "Additional STEM technicians estimated as needed to meet employer demand",
        "Additional STEM technicians estimated to be needed to meet employer demand"
    ),

    make_equivalence_pair(
        "Rise in STEM A level examination entries compared with the previous year",
        "Rise in STEM A level examination entries"
    ),

    make_equivalence_pair(
        "There is no universally accepted definition of STEM in either education or employment",
        "Definition of STEM"
    ),

    make_equivalence_pair(
        "DfE is responsible for the majority of STEM skills interventions and BEIS has a cross-cutting role",
        "Departmental responsibilities for STEM skills"
    ),

    make_equivalence_pair(
        "The absence of a precise understanding of the STEM skills problem means the efforts of DfE and BEIS are not well prioritised and a better targeted approach is needed to demonstrate value for money",
        "DfE and BEIS need a shared vision, coordinated plans and a better targeted approach to demonstrate value for money"
    ),

    make_equivalence_pair(
        "The impact of exit from the EU is difficult to predict",
        "Impact of exit from the EU is difficult to predict"
    ),

    make_equivalence_pair(
        "Government does not have a stable and consistent set of definitions for STEM in either an educational or a work context",
        "Government does not have a stable and consistent set of definitions for STEM"
    ),

    make_equivalence_pair(
        "Configure the labour market intelligence generated by Skills Advisory Panels and other mechanisms so that it enables effective decision-making",
        "Configure labour market intelligence so that it enables effective decision-making"
    ),

    make_equivalence_pair(
        "Provide departments with clarity on the different STEM definitions used in different contexts, and reasons for these different definitions",
        "Provide departments with clarity on the different STEM definitions used in different contexts and reasons for these different definitions"
    ),

    make_equivalence_pair(
        "Strengthen its work to evaluate and identify what is effective in its activities to promote participation in STEM education and skills development, and ensure this is shared with its delivery partners",
        "Strengthen work to evaluate and identify what is effective in activities to promote participation in STEM education and skills development and share this with delivery partners"
    ),

    make_equivalence_pair(
        "Working with other departments, use data on skills mismatches resulting from EU exit to establish the position across relevant sectors and determine whether key capabilities are at risk",
        "Use data on skills mismatches resulting from EU exit to establish the position across relevant sectors and determine whether key capabilities are at risk"
    ),

    make_equivalence_pair(
        "The key routes for developing STEM knowledge and skills are schools and sixth-form colleges, further education colleges, apprenticeships and higher education institutions",
        "Main STEM skills-development routes"
    )
}


def canonical_metric(value):
    return normalise_text(value)


def is_metric_equivalent(
    reference_value,
    extracted_value
):
    if (
        canonical_metric(reference_value)
        == canonical_metric(extracted_value)
    ):
        return True

    return pair_is_controlled_equivalent(
        reference_value,
        extracted_value,
        CONTROLLED_METRIC_EQUIVALENCE_PAIRS
    )


# ------------------------------------------------------------
# 7.7 Topic equivalence
# ------------------------------------------------------------

CONTROLLED_TOPIC_EQUIVALENCE_PAIRS = {
    make_equivalence_pair(
        "STEM A level initiatives",
        "A levels"
    ),

    make_equivalence_pair(
        "Female participation in STEM A levels",
        "Female students — STEM A level exam entries"
    ),

    make_equivalence_pair(
        "Female participation in computing",
        "Female students — computing"
    ),

    make_equivalence_pair(
        "Female participation in physics",
        "Female students — physics"
    ),

    make_equivalence_pair(
        "Female participation in mathematics",
        "Female students — mathematics"
    ),

    make_equivalence_pair(
        "Female participation in STEM apprenticeships",
        "Female apprentices — STEM apprenticeships"
    ),

    make_equivalence_pair(
        "Female participation in all apprenticeships",
        "Female apprentices — all apprenticeship starts"
    ),

    make_equivalence_pair(
        "Female participation in undergraduate STEM",
        "Female students — undergraduate STEM courses"
    ),

    make_equivalence_pair(
        "Female participation in all undergraduate courses",
        "Female students — all enrolments"
    ),

    make_equivalence_pair(
        "Further education STEM participation",
        "STEM further education learning aims"
    ),

    make_equivalence_pair(
        "Full-time undergraduate STEM participation",
        "Full-time STEM degrees"
    ),

    make_equivalence_pair(
        "All-subject undergraduate participation",
        "Full-time degrees — all subjects"
    ),

    make_equivalence_pair(
        "Part-time undergraduate STEM participation",
        "Part-time undergraduate STEM courses"
    ),

    make_equivalence_pair(
        "Part-time undergraduate participation",
        "Part-time degree enrolments"
    ),

    make_equivalence_pair(
        "STEM graduate destinations",
        "STEM graduates"
    ),

    make_equivalence_pair(
        "T levels",
        "Technical levels (T levels)"
    ),

    make_equivalence_pair(
        "National colleges",
        "National colleges programme"
    ),

    make_equivalence_pair(
        "Teacher supply",
        "Maths and physics teacher supply"
    ),

    make_equivalence_pair(
        "Teacher recruitment",
        "Maths and physics teachers"
    ),

    make_equivalence_pair(
        "Teacher development",
        "Non-specialist maths and physics teachers"
    ),

    make_equivalence_pair(
        "STEM interventions",
        "Key STEM-specific interventions"
    ),

    make_equivalence_pair(
        "Undergraduate STEM participation",
        "Undergraduate STEM subjects"
    ),

    make_equivalence_pair(
        "STEM graduate destinations",
        "Graduates in STEM subjects"
    ),

    make_equivalence_pair(
        "STEM technician demand",
        "STEM technicians"
    ),

    make_equivalence_pair(
        "Female participation in STEM apprenticeships",
        "Women starting STEM apprenticeships"
    ),

    make_equivalence_pair(
        "STEM A level entries",
        "STEM A level examination entries"
    ),

    make_equivalence_pair(
        "Part-time undergraduate STEM participation",
        "Part-time undergraduate STEM degrees"
    ),

    make_equivalence_pair(
        "Definition of STEM",
        "STEM in education and employment"
    ),

    make_equivalence_pair(
        "Departmental responsibility",
        "DfE, BEIS and other government departments"
    ),

    make_equivalence_pair(
        "Value for money",
        "Government approach to improving STEM skills"
    ),

    make_equivalence_pair(
        "Labour market intelligence",
        "STEM skills intelligence"
    ),

    make_equivalence_pair(
        "Cross-government coordination",
        "Government coordination on STEM"
    ),

    make_equivalence_pair(
        "EU exit and STEM skills",
        "EU exit and availability of STEM skills"
    ),

    make_equivalence_pair(
        "Estimates of STEM skills needs",
        "STEM skills estimates"
    ),

    make_equivalence_pair(
        "STEM definitions",
        "STEM definitions in educational and work contexts"
    ),

    make_equivalence_pair(
        "Labour market intelligence",
        "Skills Advisory Panels and other labour market intelligence mechanisms"
    ),

    make_equivalence_pair(
        "Evaluation and knowledge sharing",
        "STEM education and skills development activities"
    ),

    make_equivalence_pair(
        "EU exit and STEM skills",
        "EU exit and skills mismatches"
    ),

    make_equivalence_pair(
        "Skills marketplace",
        "STEM skills marketplace"
    ),

    make_equivalence_pair(
        "Cross-government coordination",
        "Cross-government approach to STEM"
    ),

    make_equivalence_pair(
        "STEM skills pipeline",
        "Schools and sixth-form colleges; further education colleges; apprenticeships; higher education institutions"
    )
}


def canonical_topic(value):
    return normalise_text(value)


def is_topic_equivalent(
    reference_value,
    extracted_value
):
    if (
        canonical_topic(reference_value)
        == canonical_topic(extracted_value)
    ):
        return True

    return pair_is_controlled_equivalent(
        reference_value,
        extracted_value,
        CONTROLLED_TOPIC_EQUIVALENCE_PAIRS
    )


print(
    "Frozen D7 source-grounded equivalence rules loaded."
)

print(
    "Metric equivalence pairs:",
    len(CONTROLLED_METRIC_EQUIVALENCE_PAIRS)
)

print(
    "Topic equivalence pairs:",
    len(CONTROLLED_TOPIC_EQUIVALENCE_PAIRS)
)


In [ ]:
# ============================================================
# 8. Null-safe numeric comparison
# ============================================================
def numeric_value(value):
    if is_null(value) or isinstance(value, bool):
        return None

    if isinstance(value, (int, float)):
        return float(value)

    if isinstance(value, str):
        text = (
            value.strip()
            .replace(",", "")
            .replace("−", "-")
            .replace("–", "-")
        )

        if re.fullmatch(
            r"-?\d+(?:\.\d+)?",
            text
        ):
            return float(text)

    return None


def numeric_exact_match(
    reference_value,
    extracted_value,
    tolerance=1e-9
):
    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if (
        reference_number is None
        and extracted_number is None
    ):
        return True

    if (
        reference_number is None
        or extracted_number is None
    ):
        return False

    return math.isclose(
        reference_number,
        extracted_number,
        rel_tol=tolerance,
        abs_tol=tolerance
    )


def numeric_absolute_match(
    reference_value,
    extracted_value,
    tolerance=1e-9
):
    reference_number = numeric_value(reference_value)
    extracted_number = numeric_value(extracted_value)

    if (
        reference_number is None
        and extracted_number is None
    ):
        return True

    if (
        reference_number is None
        or extracted_number is None
    ):
        return False

    return math.isclose(
        abs(reference_number),
        abs(extracted_number),
        rel_tol=tolerance,
        abs_tol=tolerance
    )


In [ ]:
# ============================================================
# 9. Prepare matching blocks
# ============================================================

reference_comparison_df["_reference_index"] = range(
    len(reference_comparison_df)
)

extracted_comparison_df["_extraction_index"] = range(
    len(extracted_comparison_df)
)

for dataframe in [
    reference_comparison_df,
    extracted_comparison_df
]:
    dataframe["_block_category"] = (
        dataframe["Category"].map(normalise_text)
    )

    dataframe["_block_source"] = (
        dataframe["Source Location"].map(normalise_text)
    )

    dataframe["_matching_block"] = list(
        zip(
            dataframe["_block_category"],
            dataframe["_block_source"]
        )
    )

print("Comparison copies prepared.")
print("Blocking fields:", ALIGNMENT_BLOCK_FIELDS)


In [ ]:
# ============================================================
# 10. Identity-only matching score
# ============================================================

def matching_score(
    reference_row,
    extracted_row
):

    if is_metric_equivalent(
        reference_row["Metric"],
        extracted_row["Metric"]
    ):
        metric_score = 1.0

    else:
        metric_score = text_similarity(
            reference_row["Metric"],
            extracted_row["Metric"]
        )

    # --------------------------------------------------------
    # Topic
    # --------------------------------------------------------
    if is_topic_equivalent(
        reference_row["Topic"],
        extracted_row["Topic"]
    ):
        topic_score = 1.0

    else:
        topic_score = text_similarity(
            reference_row["Topic"],
            extracted_row["Topic"]
        )

    # --------------------------------------------------------
    # Statement / Section
    # --------------------------------------------------------
    if is_statement_equivalent(
        reference_row["Statement or Section"],
        extracted_row["Statement or Section"],
        reference_row["Source Location"]
    ):
        statement_score = 1.0

    else:
        statement_score = text_similarity(
            reference_row["Statement or Section"],
            extracted_row["Statement or Section"]
        )

    # --------------------------------------------------------
    # Reporting Period
    # --------------------------------------------------------
    if is_period_equivalent(
        reference_row["Reporting Period"],
        extracted_row["Reporting Period"]
    ):
        period_score = 1.0

    else:
        period_score = text_similarity(
            reference_row["Reporting Period"],
            extracted_row["Reporting Period"]
        )

    # --------------------------------------------------------
    # Weighted identity score
    # --------------------------------------------------------
    total_score = (
        MATCHING_WEIGHTS["metric"]
        * metric_score

        + MATCHING_WEIGHTS["topic"]
        * topic_score

        + MATCHING_WEIGHTS[
            "statement_or_section"
        ]
        * statement_score

        + MATCHING_WEIGHTS[
            "reporting_period"
        ]
        * period_score
    )

    return {
        "total": total_score,
        "metric": metric_score,
        "topic": topic_score,
        "statement": statement_score,
        "period": period_score
    }


print(
    "Matching uses Category + Source Location blocking "
    "and identity/context fields only."
)

print(
    "Value, Unit and Qualifier are excluded from alignment."
)

print(
    "Controlled D7 Metric, Topic and Statement equivalences "
    "may support identity matching but do not use outcome values."
)


In [ ]:
# ============================================================
# 11. One-to-one Hungarian record alignment
# ============================================================

all_blocks = sorted(
    set(
        reference_comparison_df[
            "_matching_block"
        ]
    )
    | set(
        extracted_comparison_df[
            "_matching_block"
        ]
    ),
    key=str
)

matched_pairs = []

unmatched_reference_indices = set(
    reference_comparison_df[
        "_reference_index"
    ].tolist()
)

unmatched_extraction_indices = set(
    extracted_comparison_df[
        "_extraction_index"
    ].tolist()
)

for block in all_blocks:

    reference_block = (
        reference_comparison_df.loc[
            reference_comparison_df[
                "_matching_block"
            ] == block
        ]
    )

    extraction_block = (
        extracted_comparison_df.loc[
            extracted_comparison_df[
                "_matching_block"
            ] == block
        ]
    )

    if reference_block.empty or extraction_block.empty:
        continue

    reference_rows = [
        row
        for _, row in reference_block.iterrows()
    ]

    extraction_rows = [
        row
        for _, row in extraction_block.iterrows()
    ]

    score_matrix = []
    details_matrix = []

    for reference_row in reference_rows:
        score_row = []
        details_row = []

        for extracted_row in extraction_rows:
            details = matching_score(
                reference_row,
                extracted_row
            )

            score_row.append(details["total"])
            details_row.append(details)

        score_matrix.append(score_row)
        details_matrix.append(details_row)

    cost_matrix = [
        [
            1.0 - score
            for score in row
        ]
        for row in score_matrix
    ]

    row_positions, column_positions = (
        linear_sum_assignment(cost_matrix)
    )

    for row_position, column_position in zip(
        row_positions,
        column_positions
    ):
        details = details_matrix[
            row_position
        ][
            column_position
        ]

        if details["total"] < MATCH_SCORE_THRESHOLD:
            continue

        reference_index = int(
            reference_rows[
                row_position
            ][
                "_reference_index"
            ]
        )

        extraction_index = int(
            extraction_rows[
                column_position
            ][
                "_extraction_index"
            ]
        )

        matched_pairs.append(
            {
                "reference_index": reference_index,
                "extraction_index": extraction_index,
                "matching_score": details["total"],
                "metric_matching_score": details["metric"],
                "topic_matching_score": details["topic"],
                "statement_matching_score": details["statement"],
                "period_matching_score": details["period"]
            }
        )

        unmatched_reference_indices.discard(
            reference_index
        )

        unmatched_extraction_indices.discard(
            extraction_index
        )


aligned_record_count = len(matched_pairs)
missing_record_count = len(unmatched_reference_indices)
unsupported_record_count = len(unmatched_extraction_indices)

print("Aligned records:", aligned_record_count)
print("Missing records:", missing_record_count)
print("Unsupported/unmatched extracted records:", unsupported_record_count)


In [ ]:
# ============================================================
# 12. Missing and unsupported/unmatched record tables
# ============================================================

missing_records_df = (
    reference_comparison_df.loc[
        reference_comparison_df[
            "_reference_index"
        ].isin(
            unmatched_reference_indices
        ),
        FIELDS
    ].copy()
)

unsupported_records_df = (
    extracted_comparison_df.loc[
        extracted_comparison_df[
            "_extraction_index"
        ].isin(
            unmatched_extraction_indices
        ),
        FIELDS + ["_extraction_index"]
    ].copy()
)

print("Missing:", len(missing_records_df))
print("Unsupported/unmatched:", len(unsupported_records_df))

if not missing_records_df.empty:
    display(missing_records_df)

if not unsupported_records_df.empty:
    display(unsupported_records_df)


In [ ]:
# ============================================================
# 13. Field-level comparison of aligned records
# ============================================================

def exact_text_match(
    first,
    second
):
    return (
        normalise_text(first)
        == normalise_text(second)
    )


comparison_rows = []

for pair in matched_pairs:

    reference_row = (
        reference_comparison_df.loc[
            reference_comparison_df[
                "_reference_index"
            ] == pair["reference_index"]
        ].iloc[0]
    )

    extracted_row = (
        extracted_comparison_df.loc[
            extracted_comparison_df[
                "_extraction_index"
            ] == pair["extraction_index"]
        ].iloc[0]
    )

    # --------------------------------------------------------
    # Diagnostic lexical similarities
    # --------------------------------------------------------

    metric_similarity = text_similarity(
        reference_row["Metric"],
        extracted_row["Metric"]
    )

    topic_similarity = text_similarity(
        reference_row["Topic"],
        extracted_row["Topic"]
    )

    # --------------------------------------------------------
    # Value comparison
    # --------------------------------------------------------

    value_match = numeric_exact_match(
        reference_row["Value"],
        extracted_row["Value"]
    )

    value_absolute_match = numeric_absolute_match(
        reference_row["Value"],
        extracted_row["Value"]
    )

    sign_only_difference = (
        not value_match
        and value_absolute_match
    )

    # --------------------------------------------------------
    # Controlled D7 field comparison
    # --------------------------------------------------------

    field_matches = {
        "Category": exact_text_match(
            reference_row["Category"],
            extracted_row["Category"]
        ),

        "Statement or Section":
            is_statement_equivalent(
                reference_row[
                    "Statement or Section"
                ],
                extracted_row[
                    "Statement or Section"
                ],
                reference_row[
                    "Source Location"
                ]
            ),

        "Metric": is_metric_equivalent(
            reference_row["Metric"],
            extracted_row["Metric"]
        ),

        "Topic": is_topic_equivalent(
            reference_row["Topic"],
            extracted_row["Topic"]
        ),

        "Value": value_match,

        "Unit": is_unit_equivalent(
            reference_row["Unit"],
            extracted_row["Unit"]
        ),

        "Qualifier": is_qualifier_equivalent(
            reference_row["Qualifier"],
            extracted_row["Qualifier"]
        ),

        "Reporting Period":
            is_period_equivalent(
                reference_row[
                    "Reporting Period"
                ],
                extracted_row[
                    "Reporting Period"
                ]
            ),

        "Source Location": exact_text_match(
            reference_row[
                "Source Location"
            ],
            extracted_row[
                "Source Location"
            ]
        )
    }

    # --------------------------------------------------------
    # Discrepancy classification
    # --------------------------------------------------------

    all_mismatched_fields = [
        field
        for field in FIELDS
        if not field_matches[field]
    ]

    primary_mismatched_fields = [
        field
        for field in PRIMARY_CORRECTNESS_FIELDS
        if not field_matches[field]
    ]

    fully_correct = (
        len(primary_mismatched_fields) == 0
    )

    # --------------------------------------------------------
    # Identify where controlled equivalence was required
    # --------------------------------------------------------

    metric_rule_applied = (
        not exact_text_match(
            reference_row["Metric"],
            extracted_row["Metric"]
        )
        and is_metric_equivalent(
            reference_row["Metric"],
            extracted_row["Metric"]
        )
    )

    topic_rule_applied = (
        not exact_text_match(
            reference_row["Topic"],
            extracted_row["Topic"]
        )
        and is_topic_equivalent(
            reference_row["Topic"],
            extracted_row["Topic"]
        )
    )

    statement_rule_applied = (
        not (
            canonical_statement(
                reference_row[
                    "Statement or Section"
                ]
            )
            ==
            canonical_statement(
                extracted_row[
                    "Statement or Section"
                ]
            )
        )
        and is_statement_equivalent(
            reference_row[
                "Statement or Section"
            ],
            extracted_row[
                "Statement or Section"
            ],
            reference_row[
                "Source Location"
            ]
        )
    )

    unit_rule_applied = (
        not (
            canonical_unit(
                reference_row["Unit"]
            )
            ==
            canonical_unit(
                extracted_row["Unit"]
            )
        )
        and is_unit_equivalent(
            reference_row["Unit"],
            extracted_row["Unit"]
        )
    )

    qualifier_rule_applied = (
        not (
            canonical_qualifier(
                reference_row["Qualifier"]
            )
            ==
            canonical_qualifier(
                extracted_row["Qualifier"]
            )
        )
        and is_qualifier_equivalent(
            reference_row["Qualifier"],
            extracted_row["Qualifier"]
        )
    )

    period_rule_applied = (
        normalise_text(
            reference_row[
                "Reporting Period"
            ]
        )
        !=
        normalise_text(
            extracted_row[
                "Reporting Period"
            ]
        )
        and is_period_equivalent(
            reference_row[
                "Reporting Period"
            ],
            extracted_row[
                "Reporting Period"
            ]
        )
    )

    # --------------------------------------------------------
    # Detailed output row
    # --------------------------------------------------------

    output_row = {
        "Reference Index":
            pair["reference_index"],

        "Extraction Index":
            pair["extraction_index"],

        "Category":
            reference_row["Category"],

        "Matching Score":
            pair["matching_score"],

        "Metric Matching Score":
            pair["metric_matching_score"],

        "Topic Matching Score":
            pair["topic_matching_score"],

        "Statement Matching Score":
            pair["statement_matching_score"],

        "Reporting Period Matching Score":
            pair["period_matching_score"],

        "Metric Lexical Similarity":
            metric_similarity,

        "Topic Lexical Similarity":
            topic_similarity,

        "Metric Equivalence Rule Applied":
            bool(metric_rule_applied),

        "Topic Equivalence Rule Applied":
            bool(topic_rule_applied),

        "Statement Equivalence Rule Applied":
            bool(statement_rule_applied),

        "Unit Equivalence Rule Applied":
            bool(unit_rule_applied),

        "Qualifier Equivalence Rule Applied":
            bool(qualifier_rule_applied),

        "Reporting Period Equivalence Rule Applied":
            bool(period_rule_applied),

        "Value Sign-Only Difference":
            bool(sign_only_difference),

        "Fully Correct":
            bool(fully_correct),

        "all_mismatched_fields":
            ", ".join(
                all_mismatched_fields
            ),

        "primary_mismatched_fields":
            ", ".join(
                primary_mismatched_fields
            )
    }

    # --------------------------------------------------------
    # Preserve reference/extracted values + match flags
    # --------------------------------------------------------

    for field in FIELDS:

        output_row[
            f"Reference {field}"
        ] = reference_row[field]

        output_row[
            f"Extracted {field}"
        ] = extracted_row[field]

        output_row[
            f"{field} Match"
        ] = bool(
            field_matches[field]
        )

    comparison_rows.append(
        output_row
    )


comparison_df = pd.DataFrame(
    comparison_rows
)

print(
    "Compared aligned records:",
    len(comparison_df)
)

print(
    "Fully correct:",
    int(
        comparison_df[
            "Fully Correct"
        ].sum()
    )
    if not comparison_df.empty
    else 0
)

print(
    "Sign-only value differences:",
    int(
        comparison_df[
            "Value Sign-Only Difference"
        ].sum()
    )
    if not comparison_df.empty
    else 0
)

if not comparison_df.empty:

    print(
        "Metric equivalence rules applied:",
        int(
            comparison_df[
                "Metric Equivalence Rule Applied"
            ].sum()
        )
    )

    print(
        "Topic equivalence rules applied:",
        int(
            comparison_df[
                "Topic Equivalence Rule Applied"
            ].sum()
        )
    )

    print(
        "Statement equivalence rules applied:",
        int(
            comparison_df[
                "Statement Equivalence Rule Applied"
            ].sum()
        )
    )

    print(
        "Unit equivalence rules applied:",
        int(
            comparison_df[
                "Unit Equivalence Rule Applied"
            ].sum()
        )
    )

    print(
        "Qualifier equivalence rules applied:",
        int(
            comparison_df[
                "Qualifier Equivalence Rule Applied"
            ].sum()
        )
    )

    print(
        "Reporting-period equivalence rules applied:",
        int(
            comparison_df[
                "Reporting Period Equivalence Rule Applied"
            ].sum()
        )
    )

display(
    comparison_df.head(10)
)


In [ ]:
# ============================================================
# 14. Split fully correct and discrepant aligned records
# ============================================================
if comparison_df.empty:
    fully_correct_records_df = comparison_df.copy()
    discrepant_records_df = comparison_df.copy()

else:
    fully_correct_records_df = (
        comparison_df.loc[
            comparison_df["Fully Correct"]
        ].copy()
    )

    discrepant_records_df = (
        comparison_df.loc[
            ~comparison_df["Fully Correct"]
        ].copy()
    )

fully_correct_record_count = len(
    fully_correct_records_df
)

discrepant_record_count = len(
    discrepant_records_df
)

print("Fully correct records:", fully_correct_record_count)
print("Discrepant aligned records:", discrepant_record_count)

if not discrepant_records_df.empty:
    display(
        discrepant_records_df[
            [
                "Reference Index",
                "Extraction Index",
                "Reference Metric",
                "Extracted Metric",
                "Metric Lexical Similarity",
                "Reference Topic",
                "Extracted Topic",
                "Topic Lexical Similarity",
                "Value Sign-Only Difference",
                "primary_mismatched_fields"
            ]
        ]
    )


In [ ]:
# ============================================================
# 15. Field-level validation and error summary
# ============================================================

field_validation_rows = []

for field in FIELDS:

    match_column = f"{field} Match"

    correct_count = (
        int(comparison_df[match_column].sum())
        if not comparison_df.empty
        else 0
    )

    aligned_count = len(comparison_df)

    accuracy = (
        correct_count / aligned_count
        if aligned_count
        else None
    )

    field_validation_rows.append(
        {
            "Field": field,
            "used_in_alignment_block":
                field in ALIGNMENT_BLOCK_FIELDS,
            "used_in_matching_identity":
                field in MATCHING_IDENTITY_FIELDS,
            "used_in_primary_correctness":
                field in PRIMARY_CORRECTNESS_FIELDS,
            "Aligned Records": aligned_count,
            "Correct": correct_count,
            "Incorrect": aligned_count - correct_count,
            "Accuracy": accuracy,
            "Overall Accuracy Against Reference": (
                correct_count
                / len(reference_comparison_df)
                if len(reference_comparison_df)
                else None
            )
        }
    )

field_validation_df = pd.DataFrame(
    field_validation_rows
)

field_error_summary_df = (
    field_validation_df[
        [
            "Field",
            "used_in_alignment_block",
            "used_in_matching_identity",
            "used_in_primary_correctness",
            "Incorrect",
            "Accuracy",
            "Overall Accuracy Against Reference"
        ]
    ].copy()
)

display(field_validation_df)


In [ ]:
# ============================================================
# 16. Calculate common validation metrics
# ============================================================

reference_record_count = len(reference_comparison_df)
extracted_record_count = len(extracted_comparison_df)

fully_correct_record_count = int(
    comparison_df["Fully Correct"].sum()
) if not comparison_df.empty else 0

discrepant_record_count = (
    aligned_record_count
    - fully_correct_record_count
)

completeness = (
    aligned_record_count / reference_record_count
    if reference_record_count
    else 0.0
)

missing_rate = (
    missing_record_count / reference_record_count
    if reference_record_count
    else 0.0
)

record_precision_exact = (
    fully_correct_record_count / extracted_record_count
    if extracted_record_count
    else 0.0
)

record_recall_exact = (
    fully_correct_record_count / reference_record_count
    if reference_record_count
    else 0.0
)

record_f1_exact = (
    2 * record_precision_exact * record_recall_exact
    / (record_precision_exact + record_recall_exact)
    if (record_precision_exact + record_recall_exact)
    else 0.0
)

unsupported_rate = (
    unsupported_record_count / extracted_record_count
    if extracted_record_count
    else 0.0
)

discrepancy_rate_among_aligned = (
    discrepant_record_count / aligned_record_count
    if aligned_record_count
    else 0.0
)

correct_field_instances = int(
    sum(
        comparison_df[f"{field} Match"].sum()
        for field in PRIMARY_CORRECTNESS_FIELDS
    )
) if not comparison_df.empty else 0

expected_field_instances = int(
    reference_record_count
    * len(PRIMARY_CORRECTNESS_FIELDS)
)

field_accuracy = (
    correct_field_instances / expected_field_instances
    if expected_field_instances
    else 0.0
)

sign_only_value_difference_count = (
    int(
        comparison_df[
            "Value Sign-Only Difference"
        ].sum()
    )
    if not comparison_df.empty
    else 0
)

print("Fully correct records:", fully_correct_record_count)
print("Discrepant records:", discrepant_record_count)
print("Completeness:", completeness)
print("Exact record precision:", record_precision_exact)
print("Exact record recall:", record_recall_exact)
print("Exact record F1:", record_f1_exact)
print("Field accuracy:", field_accuracy)
print(
    "Sign-only value differences:",
    sign_only_value_difference_count
)


In [ ]:
# ============================================================
# 17. Category-level metrics
# ============================================================
category_metric_rows = []

all_categories = sorted(
    set(EXPECTED_CATEGORY_COUNTS)
    | set(
        extracted_df[
            "Category"
        ].dropna().tolist()
    )
)

for category in all_categories:

    expected_records = int(
        (
            reference_df["Category"]
            == category
        ).sum()
    )

    extracted_records_in_category = int(
        (
            extracted_df["Category"]
            == category
        ).sum()
    )

    category_comparison = (
        comparison_df.loc[
            comparison_df["Reference Category"]
            == category
        ]
        if not comparison_df.empty
        else comparison_df
    )

    aligned_records_in_category = len(
        category_comparison
    )

    fully_correct_in_category = (
        int(
            category_comparison[
                "Fully Correct"
            ].sum()
        )
        if not category_comparison.empty
        else 0
    )

    discrepant_in_category = (
        aligned_records_in_category
        - fully_correct_in_category
    )

    category_completeness = (
        aligned_records_in_category
        / expected_records
        if expected_records
        else None
    )

    category_precision = (
        fully_correct_in_category
        / extracted_records_in_category
        if extracted_records_in_category
        else 0.0
    )

    category_recall = (
        fully_correct_in_category
        / expected_records
        if expected_records
        else 0.0
    )

    category_f1 = (
        2
        * category_precision
        * category_recall
        / (
            category_precision
            + category_recall
        )
        if (
            category_precision
            + category_recall
        )
        else 0.0
    )

    category_metric_rows.append(
        {
            "Category": category,
            "Expected Records": expected_records,
            "Extracted Records": extracted_records_in_category,
            "Aligned Records": aligned_records_in_category,
            "Fully Correct Records": fully_correct_in_category,
            "Discrepant Records": discrepant_in_category,
            "Completeness": category_completeness,
            "Record Precision Exact": category_precision,
            "Record Recall Exact": category_recall,
            "Record F1 Exact": category_f1
        }
    )


category_metrics_df = pd.DataFrame(
    category_metric_rows
)

display(category_metrics_df)


In [ ]:
# ============================================================
# 18. Build final Branch B validation summary
# ============================================================

field_accuracy_dictionary = {
    row["Field"]: (
        None
        if pd.isna(row["Accuracy"])
        else float(row["Accuracy"])
    )
    for _, row in field_validation_df.iterrows()
}

category_metrics_dictionary = {
    row["Category"]: {
        "expected_records":
            int(row["Expected Records"]),
        "extracted_records":
            int(row["Extracted Records"]),
        "aligned_records":
            int(row["Aligned Records"]),
        "fully_correct_records":
            int(row["Fully Correct Records"]),
        "discrepant_records":
            int(row["Discrepant Records"]),
        "completeness": (
            None
            if pd.isna(row["Completeness"])
            else float(row["Completeness"])
        ),
        "record_precision_exact":
            float(row["Record Precision Exact"]),
        "record_recall_exact":
            float(row["Record Recall Exact"]),
        "record_f1_exact":
            float(row["Record F1 Exact"])
    }
    for _, row in category_metrics_df.iterrows()
}

equivalence_rule_usage = {
    "metric":
        int(comparison_df[
            "Metric Equivalence Rule Applied"
        ].sum()) if not comparison_df.empty else 0,
    "topic":
        int(comparison_df[
            "Topic Equivalence Rule Applied"
        ].sum()) if not comparison_df.empty else 0,
    "statement_or_section":
        int(comparison_df[
            "Statement Equivalence Rule Applied"
        ].sum()) if not comparison_df.empty else 0,
    "unit":
        int(comparison_df[
            "Unit Equivalence Rule Applied"
        ].sum()) if not comparison_df.empty else 0,
    "qualifier":
        int(comparison_df[
            "Qualifier Equivalence Rule Applied"
        ].sum()) if not comparison_df.empty else 0,
    "reporting_period":
        int(comparison_df[
            "Reporting Period Equivalence Rule Applied"
        ].sum()) if not comparison_df.empty else 0
}

summary = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,

    "reference_records": int(reference_record_count),
    "extracted_records": int(extracted_record_count),
    "aligned_records": int(aligned_record_count),
    "fully_correct_records": int(fully_correct_record_count),
    "discrepant_records": int(discrepant_record_count),
    "missing_records": int(missing_record_count),
    "unsupported_extracted_records":
        int(unsupported_record_count),

    "completeness": round(float(completeness), 4),
    "missing_rate": round(float(missing_rate), 4),
    "record_precision_exact":
        round(float(record_precision_exact), 4),
    "record_recall_exact":
        round(float(record_recall_exact), 4),
    "record_f1_exact":
        round(float(record_f1_exact), 4),
    "unsupported_rate":
        round(float(unsupported_rate), 4),
    "discrepancy_rate_among_aligned":
        round(float(discrepancy_rate_among_aligned), 4),

    "field_accuracy":
        round(float(field_accuracy), 4),

    "sign_only_value_difference_count":
        int(sign_only_value_difference_count),

    "field_accuracy_among_aligned":
        field_accuracy_dictionary,

    "schema_validity":
        schema_validity,
    "schema_diagnostics":
        schema_diagnostics,
    "structurally_evaluable":
        structurally_evaluable,
    "content_diagnostics":
        content_diagnostics,

    "alignment_block_fields":
        ALIGNMENT_BLOCK_FIELDS,
    "matching_identity_fields":
        MATCHING_IDENTITY_FIELDS,
    "alignment_identity_fields":
        ALIGNMENT_IDENTITY_FIELDS,
    "primary_correctness_fields":
        PRIMARY_CORRECTNESS_FIELDS,

    "matching_rules": {
        "blocking_fields":
            ALIGNMENT_BLOCK_FIELDS,
        "identity_fields":
            MATCHING_IDENTITY_FIELDS,
        "one_to_one_assignment":
            "Hungarian linear-sum assignment",
        "matching_score_threshold":
            MATCH_SCORE_THRESHOLD,
        "matching_score_weights":
            MATCHING_WEIGHTS,
        "value_used_for_alignment":
            False,
        "unit_used_for_alignment":
            False,
        "qualifier_used_for_alignment":
            False,
        "controlled_qualitative_equivalence_used_for_identity":
            True
    },

    "comparison_rules_frozen_from_branch_A":
        True,

    "comparison_rules": {
        "raw_extraction_modified":
            False,
        "manual_correction_applied":
            False,
        "comparison_normalisation_scope":
            "Comparison copies only",
        "numeric_comparison":
            "Signed numeric equality after deterministic parsing",
        "absolute_numeric_value_for_correctness":
            False,
        "absolute_numeric_value_for_alignment":
            False,
        "sign_only_difference":
            (
                "Diagnostic only; signed Value remains "
                "authoritative for correctness"
            )
    },

    "equivalence_rule_usage":
        equivalence_rule_usage,

    "category_metrics":
        category_metrics_dictionary,

    "input_provenance":
        input_provenance
}

print(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    )
)


In [ ]:
# ============================================================
# 19. Validation integrity checks
# ============================================================

assert (
    aligned_record_count
    + missing_record_count
    == reference_record_count
)

assert (
    aligned_record_count
    + unsupported_record_count
    == extracted_record_count
)

assert (
    fully_correct_record_count
    + discrepant_record_count
    == aligned_record_count
)

for metric_name, metric_value in {
    "completeness": completeness,
    "missing_rate": missing_rate,
    "record_precision_exact": record_precision_exact,
    "record_recall_exact": record_recall_exact,
    "record_f1_exact": record_f1_exact,
    "unsupported_rate": unsupported_rate,
    "discrepancy_rate":
        discrepancy_rate_among_aligned,
    "field_accuracy": field_accuracy
}.items():

    assert (
        0.0 <= metric_value <= 1.0
    ), (
        f"Invalid {metric_name}: {metric_value}"
    )

print("Validation integrity checks passed.")


In [ ]:
# ============================================================
# 20. Export validation artefacts
# ============================================================

DETAILED_PATH = (
    OUTPUT_DIR
    / "D7_branch_B_validation_detailed.csv"
)

FULLY_CORRECT_RECORDS_PATH = (
    OUTPUT_DIR
    / "D7_branch_B_fully_correct_records.csv"
)

DISCREPANT_RECORDS_PATH = (
    OUTPUT_DIR
    / "D7_branch_B_discrepant_records.csv"
)

MISSING_RECORDS_PATH = (
    OUTPUT_DIR
    / "D7_branch_B_missing_records.csv"
)

UNSUPPORTED_RECORDS_PATH = (
    OUTPUT_DIR
    / "D7_branch_B_unsupported_records.csv"
)

FIELD_VALIDATION_PATH = (
    OUTPUT_DIR
    / "D7_branch_B_field_validation.csv"
)

FIELD_ERROR_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D7_branch_B_field_error_summary.csv"
)

CATEGORY_METRICS_PATH = (
    OUTPUT_DIR
    / "D7_branch_B_category_metrics.csv"
)

SUMMARY_PATH = (
    OUTPUT_DIR
    / "D7_branch_B_validation_summary.json"
)

comparison_df.to_csv(
    DETAILED_PATH,
    index=False,
    encoding="utf-8-sig"
)

fully_correct_records_df.to_csv(
    FULLY_CORRECT_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

discrepant_records_df.to_csv(
    DISCREPANT_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

missing_records_df.to_csv(
    MISSING_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

unsupported_records_df.to_csv(
    UNSUPPORTED_RECORDS_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_validation_df.to_csv(
    FIELD_VALIDATION_PATH,
    index=False,
    encoding="utf-8-sig"
)

field_error_summary_df.to_csv(
    FIELD_ERROR_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig"
)

category_metrics_df.to_csv(
    CATEGORY_METRICS_PATH,
    index=False,
    encoding="utf-8-sig"
)

with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("D7 Validation B artefacts saved.")


In [ ]:
# ============================================================
# 21. Download generated validation artefacts
# ============================================================

GENERATED_OUTPUTS = [
    DETAILED_PATH,
    FULLY_CORRECT_RECORDS_PATH,
    DISCREPANT_RECORDS_PATH,
    MISSING_RECORDS_PATH,
    UNSUPPORTED_RECORDS_PATH,
    FIELD_VALIDATION_PATH,
    FIELD_ERROR_SUMMARY_PATH,
    CATEGORY_METRICS_PATH,
    SUMMARY_PATH
]

for output_path in GENERATED_OUTPUTS:
    print(
        "-",
        output_path.name,
        "| exists:",
        output_path.exists()
    )

for output_path in GENERATED_OUTPUTS:
    if output_path.exists():
        files.download(output_path)
